# Week-6 Spark Assignment

## Submitted By

**Name:** Sanskruti Dnyaneshwar Shinde

**College:** Sanjivani College of Engineering, Kopargaon

**CEI ID:** CT_CSI_DE_1174

**Technology:** Apache Spark (PySpark)

**Assignment:** Week-6 – Spark Architecture and Data Processing

**Topics Covered:**
- Spark Architecture
- Lazy Evaluation and DAG
- DataFrame Operations
- Schema Handling
- Transformations and Actions
- Wide Transformations and Shuffle
- Predicate Pushdown
- CSV vs Parquet
- Data Processing Pipeline
- Performance Best Practices

In [27]:
!pip install pyspark -q

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SuperstoreSparkAssignment") \
    .master("local[*]") \
    .getOrCreate()

spark

### STEP 1 — Spark Architecture

In [3]:
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 4.0.3
Master: local[*]
Default parallelism: 2


## Spark Architecture

Apache Spark follows a master-worker architecture.

- **Driver Program:** Controls the application, creates the SparkSession, schedules tasks, and coordinates execution.
- **Cluster Manager:** Allocates resources to Spark applications (Standalone, YARN, Mesos, Kubernetes).
- **Executors:** Worker processes that execute tasks and store data in memory.

### Execution Modes
- **Local Mode:** Runs Spark on a single machine for development and testing.
- **Client Mode:** Driver runs on the client machine while executors run on the cluster.
- **Cluster Mode:** Both the driver and executors run inside the cluster, suitable for production environments.

In this notebook, Spark is running in **Local Mode**.

STEP 2 — Upload Data & Read with Explicit Schema

In [4]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [7]:
import os
print(os.listdir("/content"))

['.config', 'train.csv', 'sample_data']


In [11]:
import pandas as pd
preview = pd.read_csv("/content/train.csv")
print(preview.shape)
print(preview.columns.tolist())
preview.head()

(9800, 18)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## Schema Handling

A predefined schema is used while reading the CSV file instead of relying on schema inference.

Benefits:
- Faster file loading
- Better performance
- Correct data types
- Reduced parsing overhead

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Row ID", IntegerType(), True),
    StructField("Order ID", StringType(), True),
    StructField("Order Date", StringType(), True),
    StructField("Ship Date", StringType(), True),
    StructField("Ship Mode", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Customer Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Product ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub-Category", StringType(), True),
    StructField("Product Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
])

df = spark.read.csv("/content/train.csv", header=True, schema=schema, encoding="ISO-8859-1")
df.printSchema()
df.show(5)

In [13]:
df_selected = df.select(
    "Order ID", "Order Date", "Customer Name", "Segment",
    "Region", "Category", "Sub-Category", "Sales"
)

# Filter: Technology orders with Sales above 200
df_filtered = df_selected.filter(
    (df_selected["Category"] == "Technology") & (df_selected["Sales"] > 200)
)

df_filtered.show(5)
print("Filtered row count:", df_filtered.count())

+--------------+----------+------------------+-----------+-------+----------+------------+--------+
|      Order ID|Order Date|     Customer Name|    Segment| Region|  Category|Sub-Category|   Sales|
+--------------+----------+------------------+-----------+-------+----------+------------+--------+
|CA-2015-115812|09/06/2015|   Brosina Hoffman|   Consumer|   West|Technology|      Phones| 907.152|
|CA-2015-115812|09/06/2015|   Brosina Hoffman|   Consumer|   West|Technology|      Phones| 911.424|
|CA-2015-143336|27/08/2015|Zuschuss Donatelli|   Consumer|   West|Technology|      Phones|  213.48|
|CA-2017-117590|08/12/2017|         Gene Hale|  Corporate|Central|Technology|      Phones|1097.544|
|CA-2016-117415|27/12/2016|      Steve Nguyen|Home Office|Central|Technology|      Phones| 371.168|
+--------------+----------+------------------+-----------+-------+----------+------------+--------+
only showing top 5 rows
Filtered row count: 835


## Filtering and Column Selection

Filtering removes unnecessary records before processing, reducing the amount of data Spark needs to process.

The `select()` operation retrieves only the required columns, minimizing memory usage and improving execution performance.

These are examples of narrow transformations, where data remains within the same partition without requiring data movement across executors.

In [15]:
from pyspark.sql.functions import to_date, col, when

df_renamed = df_filtered.withColumnRenamed("Sub-Category", "SubCategory") \
                         .withColumnRenamed("Order Date", "OrderDate")

df_casted = df_renamed.withColumn("OrderDate", to_date(col("OrderDate"), "dd/MM/yyyy"))

df_final = df_casted.withColumn(
    "HighValueOrder", when(col("Sales") > 500, "Yes").otherwise("No")
)

df_final.printSchema()
df_final.show(5)

root
 |-- Order ID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- HighValueOrder: string (nullable = false)

+--------------+----------+------------------+-----------+-------+----------+-----------+--------+--------------+
|      Order ID| OrderDate|     Customer Name|    Segment| Region|  Category|SubCategory|   Sales|HighValueOrder|
+--------------+----------+------------------+-----------+-------+----------+-----------+--------+--------------+
|CA-2015-115812|2015-06-09|   Brosina Hoffman|   Consumer|   West|Technology|     Phones| 907.152|           Yes|
|CA-2015-115812|2015-06-09|   Brosina Hoffman|   Consumer|   West|Technology|     Phones| 911.424|           Yes|
|CA-2015-143336|2015-08-27|Zuschuss Donatelli|   Consume

## DataFrame Transformations

The DataFrame was modified using several transformation operations:

- Renamed columns using `withColumnRenamed()`.
- Added new columns using `withColumn()`.
- Converted data types using `cast()` and `to_date()`.

Spark DataFrames are immutable, meaning each transformation creates a new DataFrame without modifying the original one.

In [16]:
df_final.select("OrderDate").show(5)

+----------+
| OrderDate|
+----------+
|2015-06-09|
|2015-06-09|
|2015-08-27|
|2017-12-08|
|2016-12-27|
+----------+
only showing top 5 rows


## Handling Missing Values

Missing values can reduce data quality and affect analysis.

In this assignment, null values were identified using Spark SQL functions.

Spark also provides methods such as:

- `drop()`
- `fill()`
- `filter()`

to clean datasets before analysis.

In [17]:
from pyspark.sql.functions import sum as spark_sum

df_final.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_final.columns
]).show()

+--------+---------+-------------+-------+------+--------+-----------+-----+--------------+
|Order ID|OrderDate|Customer Name|Segment|Region|Category|SubCategory|Sales|HighValueOrder|
+--------+---------+-------------+-------+------+--------+-----------+-----+--------------+
|       0|        0|            0|      0|     0|       0|          0|    0|             0|
+--------+---------+-------------+-------+------+--------+-----------+-----+--------------+



## Wide Transformation and Shuffle

The `groupBy()` operation is a **Wide Transformation** because Spark redistributes data across partitions before performing aggregation.

This redistribution is called **Shuffle**.

Shuffle is an expensive operation because it involves:

- Network communication
- Disk I/O
- Data sorting

Reducing unnecessary shuffle operations improves Spark performance.

In [18]:
from pyspark.sql.functions import round as spark_round

agg_df = df_final.groupBy("Region", "Category") \
                  .agg(spark_round(spark_sum("Sales"), 2).alias("TotalSales")) \
                  .orderBy(col("TotalSales").desc())

agg_df.show()
agg_df.explain(True)   # look for "Exchange" in the plan — that's the shuffle

+-------+----------+----------+
| Region|  Category|TotalSales|
+-------+----------+----------+
|   East|Technology|  240974.8|
|   West|Technology| 219953.69|
|Central|Technology| 150462.06|
|  South|Technology| 136222.15|
+-------+----------+----------+

== Parsed Logical Plan ==
'Sort ['TotalSales DESC NULLS LAST], true
+- Aggregate [Region#54, Category#56], [Region#54, Category#56, round(sum(Sales#59), 2) AS TotalSales#356]
   +- Project [Order ID#43, OrderDate#223, Customer Name#48, Segment#49, Region#54, Category#56, SubCategory#221, Sales#59, CASE WHEN (Sales#59 > cast(500 as double)) THEN Yes ELSE No END AS HighValueOrder#224]
      +- Project [Order ID#43, to_date(OrderDate#222, Some(dd/MM/yyyy), Some(Etc/UTC), true) AS OrderDate#223, Customer Name#48, Segment#49, Region#54, Category#56, SubCategory#221, Sales#59]
         +- Project [Order ID#43, Order Date#44 AS OrderDate#222, Customer Name#48, Segment#49, Region#54, Category#56, SubCategory#221, Sales#59]
            +- Pro

In [19]:
lazy_df = df_final.filter(col("Sales") > 500).select("Category", "Sales")
lazy_df.explain()   # nothing has executed yet
lazy_df.show(5)     # NOW it executes

== Physical Plan ==
*(1) Filter ((((isnotnull(Category#56) AND isnotnull(Sales#59)) AND (Category#56 = Technology)) AND (Sales#59 > 200.0)) AND (Sales#59 > 500.0))
+- FileScan csv [Category#56,Sales#59] Batched: false, DataFilters: [isnotnull(Category#56), isnotnull(Sales#59), (Category#56 = Technology), (Sales#59 > 200.0), (Sa..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/train.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Category), IsNotNull(Sales), EqualTo(Category,Technology), GreaterThan(Sales,200.0), G..., ReadSchema: struct<Category:string,Sales:double>


+----------+--------+
|  Category|   Sales|
+----------+--------+
|Technology| 907.152|
|Technology| 911.424|
|Technology|1097.544|
|Technology| 1029.95|
|Technology|  944.93|
+----------+--------+
only showing top 5 rows


## Lazy Evaluation and DAG

Spark transformations are evaluated lazily.

Instead of executing each transformation immediately, Spark creates a Directed Acyclic Graph (DAG) representing the sequence of operations.

Execution starts only when an action such as `show()`, `count()`, or `write()` is called.

In [21]:
def run_pipeline(input_path, output_path):
    df = spark.read.csv(input_path, header=True, schema=schema, encoding="ISO-8859-1")

    pipeline_df = (
        df.select("Order ID", "Order Date", "Customer Name", "Segment",
                   "Region", "Category", "Sub-Category", "Sales")
          .withColumnRenamed("Sub-Category", "SubCategory")
          .withColumnRenamed("Order Date", "OrderDate")
          .withColumn("OrderDate", to_date(col("OrderDate"), "dd/MM/yyyy"))
          .withColumn("HighValueOrder", when(col("Sales") > 500, "Yes").otherwise("No"))
          .filter(col("Sales") > 0)
          .dropna(subset=["Sales"])
    )

    pipeline_df.write.mode("overwrite").parquet(output_path)
    print(f"Pipeline complete. Rows written: {pipeline_df.count()}")
    return pipeline_df

## Data Processing Pipeline

Pipeline Steps:

1. Read input data
2. Apply transformations
3. Filter records
4. Handle missing values
5. Save processed data to CSV
6. Save processed data to Parquet

This modular pipeline improves scalability and maintainability.

In [22]:
result_df = run_pipeline("/content/train.csv", "final_output_parquet")
result_df.show(10)

Pipeline complete. Rows written: 9508
+--------------+----------+---------------+---------+------+---------------+-----------+--------+--------------+
|      Order ID| OrderDate|  Customer Name|  Segment|Region|       Category|SubCategory|   Sales|HighValueOrder|
+--------------+----------+---------------+---------+------+---------------+-----------+--------+--------------+
|CA-2017-152156|2017-11-08|    Claire Gute| Consumer| South|      Furniture|  Bookcases|  261.96|            No|
|CA-2017-152156|2017-11-08|    Claire Gute| Consumer| South|      Furniture|     Chairs|  731.94|           Yes|
|CA-2017-138688|2017-06-12|Darrin Van Huff|Corporate|  West|Office Supplies|     Labels|   14.62|            No|
|US-2016-108966|2016-10-11| Sean O'Donnell| Consumer| South|      Furniture|     Tables|957.5775|           Yes|
|US-2016-108966|2016-10-11| Sean O'Donnell| Consumer| South|Office Supplies|    Storage|  22.368|            No|
|CA-2015-115812|2015-06-09|Brosina Hoffman| Consumer|  Wes

In [23]:
import time

start = time.time()
df_final.write.mode("overwrite").option("header", True).csv("output_csv")
csv_time = time.time() - start

start = time.time()
df_final.write.mode("overwrite").parquet("output_parquet")
parquet_time = time.time() - start

print(f"CSV write time: {csv_time:.2f}s")
print(f"Parquet write time: {parquet_time:.2f}s")

CSV write time: 0.99s
Parquet write time: 0.88s


In [24]:
start = time.time()
csv_read = spark.read.option("header", True).csv("output_csv")
csv_read.filter(col("Category") == "Technology").count()
print("CSV filtered read time:", time.time() - start)

start = time.time()
parquet_read = spark.read.parquet("output_parquet")
parquet_read.filter(col("Category") == "Technology").count()
print("Parquet filtered read time:", time.time() - start)

parquet_read.filter(col("Category") == "Technology").explain(True)

CSV filtered read time: 1.1566190719604492
Parquet filtered read time: 0.7488796710968018
== Parsed Logical Plan ==
'Filter '`=`('Category, Technology)
+- Relation [Order ID#531,OrderDate#532,Customer Name#533,Segment#534,Region#535,Category#536,SubCategory#537,Sales#538,HighValueOrder#539] parquet

== Analyzed Logical Plan ==
Order ID: string, OrderDate: date, Customer Name: string, Segment: string, Region: string, Category: string, SubCategory: string, Sales: double, HighValueOrder: string
Filter (Category#536 = Technology)
+- Relation [Order ID#531,OrderDate#532,Customer Name#533,Segment#534,Region#535,Category#536,SubCategory#537,Sales#538,HighValueOrder#539] parquet

== Optimized Logical Plan ==
Filter (isnotnull(Category#536) AND (Category#536 = Technology))
+- Relation [Order ID#531,OrderDate#532,Customer Name#533,Segment#534,Region#535,Category#536,SubCategory#537,Sales#538,HighValueOrder#539] parquet

== Physical Plan ==
*(1) Filter (isnotnull(Category#536) AND (Category#536 =

## Predicate Pushdown

Parquet stores data in a columnar format.

When filters are applied, Spark reads only the required columns and pushes filter conditions closer to the storage layer.

This optimization is called **Predicate Pushdown**, reducing disk I/O and improving query performance.

In [25]:
!du -sh output_csv output_parquet

84K	output_csv
40K	output_parquet


## CSV vs Parquet Performance

### CSV
- Row-based storage
- Larger file size
- Slower read performance
- No built-in compression

### Parquet
- Columnar storage
- Compressed format
- Faster analytical queries
- Supports Predicate Pushdown

Parquet is generally preferred for large-scale data processing because it provides better storage efficiency and faster query execution.

# Conclusion

This assignment demonstrated efficient data processing using Apache Spark.

### Key Learnings

- Understood Spark architecture and execution modes.
- Learned Lazy Evaluation and DAG execution.
- Performed filtering, selection, renaming, datatype conversion, and column creation.
- Handled missing values effectively.
- Applied narrow and wide transformations.
- Understood Shuffle and Predicate Pushdown.
- Compared CSV and Parquet formats.
- Built a complete Spark data pipeline.
- Followed Spark best practices by using explicit schemas and avoiding `collect()` on large datasets.

Overall, Spark provides a scalable and efficient framework for processing large datasets with optimized execution and high performance.